# S2 · от сенсорного лога к матрице признаков

Шесть этапов преподавательского live coding. Сначала преподаватель объясняет операцию, затем вводит её код, запускает и разбирает результат.

## Исходный лог и дефекты времени

Преподаватель загружает синтетическую запись и пишет диагностику времени IMU. Счётчики не исправляют лог: они показывают, какие дефекты в нём есть. Один `NaN` даёт один `non_finite`, хотя соседних пар нет.

In [ ]:
import json
import sys
from pathlib import Path

import numpy as np

# Jupyter обычно открывает kernel в notebooks; запуск из starter также поддержан.
STARTER_ROOT = next(
    p for p in (Path.cwd(), Path.cwd().parent)
    if (p / "src" / "ml_sau" / "sensor.py").is_file()
)
sys.path.insert(0, str(STARTER_ROOT / "src"))
from ml_sau.sensor_source import G0, generate_sensor_log
from ml_sau.sensor import (
    FEATURE_NAMES, GAP_TOLERANCE_S, WindowBatch,
    assemble_quality_report, linear_interpolation, write_sensor_artifacts,
)

raise NotImplementedError("S2 block 1: код вводит преподаватель")

assert timestamp_report(np.array([0.0, 0.1, 0.1]))["duplicate"] == 1
assert timestamp_report(np.array([0.0, 0.2, 0.1]))["backward"] == 1
assert timestamp_report(np.array([np.nan]))["non_finite"] == 1
assert timestamp_report(np.array([])) == {"non_finite": 0, "duplicate": 0, "backward": 0}
assert time_quality["non_finite"] == time_quality["backward"] == 0


## Единицы и общие часы

Два канала акселерометра задают удельную силу по осям x и z датчика, третий — угловую скорость. GNSS даёт скорость. Предоставлены параметры часов: `t_gnss = 1.00022 * t_common + 0.18`. Преподаватель переводит единицы, обращает эту формулу и строит общую сетку. Оценка параметров часов не входит в занятие.

In [ ]:
raise NotImplementedError("S2 block 2: код вводит преподаватель")

np.testing.assert_allclose(180.0 * np.pi / 180.0, np.pi)
np.testing.assert_allclose(gnss_t[[0, -1]], [0.0, 30.0])
assert np.all(np.diff(imu_t) > 0)


## Интерполяция и разрывы

Готовая `linear_interpolation` проверяет формы и выполняет `np.interp` по каналам; за пределами записи возвращает `NaN`. Преподаватель добавляет правило длинных интервалов. Допуск `1e-9` с учитывает округление чисел: интервалы GNSS номинально равны 0.10 с. Низкочастотные учебные сигналы позволяют использовать сетку 50 Гц; это не универсальный рецепт понижения частоты.

In [ ]:
raise NotImplementedError("S2 block 3: код вводит преподаватель")

# Один и тот же числовой пример: значения концов сохраняются при обоих пределах.
toy_t = np.array([0.0, 2.0])
toy_v = np.array([0.0, 4.0])
toy_grid = np.array([-1.0, 0.0, 1.0, 2.0, 3.0])
np.testing.assert_allclose(interpolate_with_gap_mask(toy_t, toy_v, toy_grid, 2.0), [np.nan, 0.0, 2.0, 4.0, np.nan])
np.testing.assert_allclose(interpolate_with_gap_mask(toy_t, toy_v, toy_grid, 0.5), [np.nan, 0.0, np.nan, 4.0, np.nan])
assert np.isfinite(speed_aligned).all()


## Окна и их происхождение

Преподаватель строит окна по 100 отсчётов с шагом 50 и сохраняет время начала всех кандидатов. Для текущего массива из 1500 строк этого достаточно; эталонная `build_window_batch` дополнительно проверяет формы и поддерживает короткие записи. На сетке 50 Гц длительность окна по числу отсчётов — 2 с; первая и последняя метки внутри него различаются на 1.98 с.

In [ ]:
raise NotImplementedError("S2 block 4: код вводит преподаватель")

np.testing.assert_allclose(batch.start_s[~batch.valid], [16.0, 17.0, 18.0])
assert batch.values.shape == (29, 100, 4)


## Признаки и матрица X

Преподаватель вычисляет четыре агрегата для каждого допустимого окна. RMS включает постоянную составляющую, стандартное отклонение вычитает среднее. Используется `ddof=0`: это характеристика данного окна. `X` ещё не является размеченной выборкой: метки режима полёта или другого целевого события потребуют отдельного источника.

In [ ]:
raise NotImplementedError("S2 block 5: код вводит преподаватель")

assert X.shape == (int(batch.valid.sum()), len(FEATURE_NAMES))
assert np.isfinite(X).all()


## Сравнение правил и сохранение результата

Преподаватель повторяет обработку с пределом 0.50 с, показывает добавленные окна и их признаки. Основными остаются артефакты варианта 0.10 с: `s2-quality.json`, `s2-features.csv`, `s2-windows.csv`. Больше строк после интерполяции не означает больше измеренной информации.

In [ ]:
raise NotImplementedError("S2 block 6: код вводит преподаватель")

# Служебная запись уже подготовлена: вычисления сделаны в предыдущих блоках.
report = assemble_quality_report(log, time_quality, grid_s, batch, X, max_gap_s)
write_sensor_artifacts(STARTER_ROOT / "reports", report, X, batch)
print(json.dumps(report, ensure_ascii=False, indent=2))
print("saved:", STARTER_ROOT / "reports" / "s2-features.csv")
